# Measure your model
Image-level text labels only. Dummy results do not establish OCR accuracy or deployment performance.

Historical plate timing uses 256 crops and max32 allocations. Current 41-class CCT64 price research separately measured0.746 ms RGB /0.937 ms encoded and 542 MiB on 144 development images, with 66/144 literal exact. CCT128 has 65/144 and unmeasured inference resources. These are not timings or accuracy promises for this public 37-class pipeline. Measure the actual exported checkpoint, preprocessing and decoder.

Set `USE_ENERGY_METER = True` if you trained the meter example.

In [ ]:
from pathlib import Path
import os, subprocess, sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
os.chdir(ROOT)
assert (ROOT / 'config/cct_s_v2_global_plate_config.yaml').exists()
def run(*args): subprocess.run(list(args), check=True)

USE_ENERGY_METER = os.environ.get("CCT_ENERGY_METER", "0") == "1"
WORK = Path(os.environ.get('CCT_WORK', 'artifacts/energy_meter' if USE_ENERGY_METER else 'artifacts'))
TRAINING = Path(os.environ.get("CCT_OUTPUT", WORK if USE_ENERGY_METER else WORK / "training"))
PLATE_CONFIG = ('config/cct_s_energy_meter_config.yaml' if USE_ENERGY_METER
                else 'config/cct_s_v2_global_plate_config.yaml')
ANNOTATIONS = ('dataset/energy-meter/val_annotations.csv' if USE_ENERGY_METER
               else 'dummy_data/val_annotations.csv')


In [ ]:
# Run only after parity passes. Dummy timing is functional, not a real benchmark.
for batch in (1,2,8,32):
    run(sys.executable,'scripts/evaluate.py','--annotations',ANNOTATIONS,'--plate-config',PLATE_CONFIG,'--backend','trt','--model',str(WORK / 'model.engine'),'--batch-size',str(batch),'--repeats','10','--output',str(WORK / f'benchmark_b{batch}.json'))
# Use a sufficiently large real evaluation CSV to fill every batch.
# Report hardware, engine precision/profile, cohort, RGB scope and sampled PID VRAM separately.